## EV Resale Price Regression

#### 1. Data Quality Audit
#### Perform a complete audit of the dataset:
#### • shape
#### • datatypes
#### • missing values
#### • duplicate records
#### • unique vehicle IDs
#### • categorical distributions
#### Identify every data-quality problem before modifying the dataset.

In [2]:
import pandas as pd

df = pd.read_csv("C:\\Users\\rohit\\Downloads\\EV_Resale_Price_Regression.csv")

print("Shape of Dataset:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nUnique Vehicle IDs:")
print(df["vehicle_id"].nunique())

categorical_cols = df.select_dtypes(include="object").columns

print("\nCategorical Distributions:")
for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts())

Shape of Dataset:
(1000, 15)

Data Types:
vehicle_id               object
listing_date             object
manufacture_year          int64
brand                    object
vehicle_type             object
battery_capacity_kwh    float64
battery_health_pct      float64
range_km                float64
km_driven               float64
charging_time_hr        float64
fast_charging            object
owner_count               int64
city                     object
service_history          object
resale_price            float64
dtype: object

Missing Values:
vehicle_id               0
listing_date             0
manufacture_year         0
brand                    0
vehicle_type             0
battery_capacity_kwh    25
battery_health_pct      30
range_km                20
km_driven                0
charging_time_hr        24
fast_charging            0
owner_count              0
city                     0
service_history         30
resale_price             0
dtype: int64

Duplicate Records:
0

Unique

#### 2. Duplicate Vehicle Investigation
#### vehicle_id is expected to uniquely identify a vehicle.
#### Determine:
#### • how many duplicated vehicle_id values exist
#### • which vehicle IDs are duplicated
#### • how many records are affected
#### Then decide how you would handle these records without blindly using drop_duplicates().

In [3]:
duplicate_mask = df["vehicle_id"].duplicated(keep=False)

print("Duplicated vehicle IDs:", df.loc[duplicate_mask, "vehicle_id"].nunique())

print("Duplicated vehicle IDs:", df.loc[duplicate_mask, "vehicle_id"].nunique())

print("\nDuplicate Vehicle IDs:")
print(df.loc[duplicate_mask, "vehicle_id"].unique())

print("\nTotal affected records:", duplicate_mask.sum())

print("\nDuplicate Records:")
print(df[duplicate_mask].sort_values("vehicle_id"))

Duplicated vehicle IDs: 6
Duplicated vehicle IDs: 6

Duplicate Vehicle IDs:
['EV-20515' 'EV-20653' 'EV-20011' 'EV-20544' 'EV-20193' 'EV-20279']

Total affected records: 12

Duplicate Records:
    vehicle_id listing_date  manufacture_year       brand vehicle_type  \
119   EV-20011   2025-04-30              2016      Nexora        Sedan   
852   EV-20011   2027-05-03              2016      Nexora        Sedan   
655   EV-20193   2026-10-18              2023     Atheron    Crossover   
797   EV-20193   2027-03-09              2023     Atheron    Crossover   
816   EV-20279   2027-03-28              2019    E-Motion          SUV   
857   EV-20279   2027-05-08              2019    E-Motion          SUV   
72    EV-20515   2025-03-14              2016  GreenDrive    Hatchback   
621   EV-20515   2026-09-14              2016  GreenDrive    Hatchback   
133   EV-20544   2025-05-14              2024    E-Motion        Sedan   
280   EV-20544   2025-10-08              2024    E-Motion        Sed

#### 3. Date Conversion & Validation
##### Convert listing_date into a proper datetime column.
##### Then investigate:
##### • earliest listing date
##### • latest listing date
##### • invalid/missing dates
##### • whether the date column is suitable for feature engineering.

In [4]:
df["listing_date"] = pd.to_datetime(df["listing_date"], errors="coerce")

print("Earliest Listing Date:")
print(df["listing_date"].min())


print("\nLatest Listing Date:")
print(df["listing_date"].max())


print("\nInvalid/Missing Dates:")
print(df["listing_date"].isnull().sum())

print("\nSuitable for Feature Engineering:",
      df["listing_date"].notnull().all())

Earliest Listing Date:
2025-01-01 00:00:00

Latest Listing Date:
2027-09-27 00:00:00

Invalid/Missing Dates:
0

Suitable for Feature Engineering: True


#### 4. Vehicle Age Feature
##### Create vehicle_age using:
##### listing year − manufacture year
##### Then identify whether any vehicle has:
##### • age < 0
##### • age = 0
##### • unusually high age

In [5]:
df["listing_date"] = pd.to_datetime(df["listing_date"], errors="coerce")

df["vehicle_age"] = df["listing_date"].dt.year - df["manufacture_year"]


print("Vehicles with age < 0:")
print(df[df["vehicle_age"] < 0])


print("\nVehicles with age = 0:")
print(df[df["vehicle_age"] == 0])

print("\nVehicles with unusually high age:")
print(df[df["vehicle_age"] > 20])

print("\nVehicle Age Summary:")
print(df["vehicle_age"].describe())

Vehicles with age < 0:
Empty DataFrame
Columns: [vehicle_id, listing_date, manufacture_year, brand, vehicle_type, battery_capacity_kwh, battery_health_pct, range_km, km_driven, charging_time_hr, fast_charging, owner_count, city, service_history, resale_price, vehicle_age]
Index: []

Vehicles with age = 0:
    vehicle_id listing_date  manufacture_year       brand vehicle_type  \
1     EV-20530   2025-01-02              2025      Nexora    Crossover   
8     EV-20708   2025-01-09              2025      Nexora        Sedan   
9     EV-20059   2025-01-10              2025     Atheron        Sedan   
12    EV-20381   2025-01-13              2025      Nexora        Sedan   
19    EV-20301   2025-01-20              2025    E-Motion        Sedan   
22    EV-20842   2025-01-23              2025     Atheron    Crossover   
31    EV-20891   2025-02-01              2025      Voltix    Hatchback   
65    EV-20356   2025-03-07              2025     Atheron    Crossover   
91    EV-20607   2025-04-02

#### 5. Battery Data Imputation
##### The following columns contain missing values:
##### battery_capacity_kwh, battery_health_pct, range_km, charging_time_hr
##### Develop an appropriate missing-value strategy for each column.
##### Do not automatically use the same statistic for every column.

##### Explain why you selected mean, median, or another strategy.

In [7]:
df["battery_capacity_kwh"] = df["battery_capacity_kwh"].fillna(df["battery_capacity_kwh"].median())

df["battery_health_pct"] = df["battery_health_pct"].fillna(df["battery_health_pct"].mean())

df["range_km"] = df["range_km"].fillna(df["range_km"].median())

df["charging_time_hr"] = df["charging_time_hr"].fillna(df["charging_time_hr"].median())

df = df.fillna({
    "battery_capacity_kwh": df["battery_capacity_kwh"].median(),
    "battery_health_pct": df["battery_health_pct"].mean(),
    "range_km": df["range_km"].median(),
    "charging_time_hr": df["charging_time_hr"].median()
})

print(df[[
    "battery_capacity_kwh",
    "battery_health_pct",
    "range_km",
    "charging_time_hr"
]].isnull().sum())

battery_capacity_kwh    0
battery_health_pct      0
range_km                0
charging_time_hr        0
dtype: int64


#### 6. Battery Health Outlier Investigation
##### Analyze battery_health_pct.
##### Identify:
##### • values below a reasonable minimum
#####  • values above 100
##### • extreme observations

In [9]:
print("Battery Health < 0:")
print(df[df["battery_health_pct"] < 0])

print("Values below 0:")
print(df[df["battery_health_pct"] < 0])

# Values above 100%
print("\nValues above 100:")
print(df[df["battery_health_pct"] > 100])

# Detect extreme observations using IQR
Q1 = df["battery_health_pct"].quantile(0.25)
Q3 = df["battery_health_pct"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("\nExtreme Observations:")
print(df[(df["battery_health_pct"] < lower_limit) |
         (df["battery_health_pct"] > upper_limit)])

print("\nLower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

Battery Health < 0:
Empty DataFrame
Columns: [vehicle_id, listing_date, manufacture_year, brand, vehicle_type, battery_capacity_kwh, battery_health_pct, range_km, km_driven, charging_time_hr, fast_charging, owner_count, city, service_history, resale_price, vehicle_age]
Index: []
Values below 0:
Empty DataFrame
Columns: [vehicle_id, listing_date, manufacture_year, brand, vehicle_type, battery_capacity_kwh, battery_health_pct, range_km, km_driven, charging_time_hr, fast_charging, owner_count, city, service_history, resale_price, vehicle_age]
Index: []

Values above 100:
Empty DataFrame
Columns: [vehicle_id, listing_date, manufacture_year, brand, vehicle_type, battery_capacity_kwh, battery_health_pct, range_km, km_driven, charging_time_hr, fast_charging, owner_count, city, service_history, resale_price, vehicle_age]
Index: []

Extreme Observations:
    vehicle_id listing_date  manufacture_year     brand vehicle_type  \
135   EV-20865   2025-05-16              2018    Nexora          SUV  

In [10]:
df["range_per_kwh"] = df["range_km"] / df["battery_capacity_kwh"]


print("Range per kWh Summary:")
print(df["range_per_kwh"].describe())

Q1 = df["range_per_kwh"].quantile(0.25)
Q3 = df["range_per_kwh"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR


print("\nVehicles with Unusually Low Efficiency:")
print(df[df["range_per_kwh"] < lower_limit])


print("\nVehicles with Unusually High Efficiency:")
print(df[df["range_per_kwh"] > upper_limit])


print("\nLower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

Range per kWh Summary:
count    1000.000000
mean        6.748555
std         2.385745
min         2.340094
25%         5.092635
50%         6.379348
75%         8.039904
max        19.200000
Name: range_per_kwh, dtype: float64

Vehicles with Unusually Low Efficiency:
Empty DataFrame
Columns: [vehicle_id, listing_date, manufacture_year, brand, vehicle_type, battery_capacity_kwh, battery_health_pct, range_km, km_driven, charging_time_hr, fast_charging, owner_count, city, service_history, resale_price, vehicle_age, range_per_kwh]
Index: []

Vehicles with Unusually High Efficiency:
    vehicle_id listing_date  manufacture_year       brand vehicle_type  \
6     EV-20961   2025-01-07              2017     Atheron        Sedan   
21    EV-20934   2025-01-22              2024  GreenDrive    Hatchback   
31    EV-20891   2025-02-01              2025      Voltix    Hatchback   
133   EV-20544   2025-05-14              2024    E-Motion        Sedan   
139   EV-20278   2025-05-20              2017

#### 8. Driving Intensity Feature
##### Create km_per_year using:
##### km_driven / vehicle_age
##### Handle the special case where vehicle_age = 0.
##### Identify unusually high annual driving.

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:\\Users\\rohit\\Downloads\\EV_Resale_Price_Regression.csv")
df["listing_date"] = pd.to_datetime(df["listing_date"], errors="coerce")

df["vehicle_age"] = df["listing_date"].dt.year - df["manufacture_year"]

df["km_per_year"] = np.where(
    df["vehicle_age"] == 0,
    np.nan,
    df["km_driven"] / df["vehicle_age"]
)

Q1 = df["km_per_year"].quantile(0.25)
Q3 = df["km_per_year"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Vehicles with unusually high annual driving:")
print(df[df["km_per_year"] > upper_limit])

print("\nSummary:")
print(df["km_per_year"].describe())

Vehicles with unusually high annual driving:
    vehicle_id listing_date  manufacture_year    brand vehicle_type  \
15    EV-20445   2025-01-16              2023   Nexora    Hatchback   
23    EV-20663   2025-01-24              2024   Nexora    Crossover   
28    EV-20997   2025-01-29              2023   Voltix        Sedan   
35    EV-20163   2025-02-05              2024  Atheron    Crossover   
44    EV-20493   2025-02-14              2024   Voltix        Sedan   
..         ...          ...               ...      ...          ...   
844   EV-20722   2027-04-25              2024  Atheron        Sedan   
864   EV-20975   2027-05-15              2025  Atheron          SUV   
886   EV-20718   2027-06-06              2025  Atheron        Sedan   
914   EV-20496   2027-07-04              2024  Atheron    Crossover   
983   EV-20062   2027-09-11              2024   Nexora        Sedan   

     battery_capacity_kwh  battery_health_pct  range_km  km_driven  \
15                   61.2       

#### 9. Charging Efficiency
##### Create charging_efficiency using:
##### range_km / charging_time_hr
##### Investigate missing values and extreme values before using this feature in a regression model.

In [13]:
import pandas as pd

df = pd.read_csv("C:\\Users\\rohit\\Downloads\\EV_Resale_Price_Regression.csv")

df["charging_efficiency"] = df["range_km"] / df["charging_time_hr"]

print("Missing Values:")
print(df["charging_efficiency"].isnull().sum())

print("\nSummary:")
print(df["charging_efficiency"].describe())

Q1 = df["charging_efficiency"].quantile(0.25)
Q3 = df["charging_efficiency"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("\nExtreme Values:")
print(df[
    (df["charging_efficiency"] < lower_limit) |
    (df["charging_efficiency"] > upper_limit)
])

print("\nLower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

Missing Values:
44

Summary:
count    956.000000
mean      66.976136
std       30.117138
min       20.888889
25%       47.017091
50%       59.742102
75%       79.061684
max      230.000000
Name: charging_efficiency, dtype: float64

Extreme Values:
    vehicle_id listing_date  manufacture_year       brand vehicle_type  \
27    EV-20085   2025-01-28              2022      Voltix        Sedan   
133   EV-20544   2025-05-14              2024    E-Motion        Sedan   
147   EV-20420   2025-05-28              2019  GreenDrive    Crossover   
166   EV-20786   2025-06-16              2018    E-Motion          SUV   
171   EV-20655   2025-06-21              2025      Voltix    Crossover   
185   EV-20355   2025-07-05              2016     Atheron    Hatchback   
243   EV-20339   2025-09-01              2024    E-Motion          SUV   
267   EV-20551   2025-09-25              2023      Voltix        Sedan   
280   EV-20544   2025-10-08              2024    E-Motion        Sedan   
327   EV-208

#### 10. Ownership Analysis
##### Analyze owner_count.
##### Determine:
##### • frequency of each ownership level
##### • whether any values are invalid
##### • whether owner count should be treated as numerical or categorical for regression.
##### Justify your decision.

In [14]:
print("Frequency of Owner Count:")
print(df["owner_count"].value_counts(dropna=False))

print("\nInvalid Values (owner_count <= 0):")
print(df[df["owner_count"] <= 0])

print("\nMissing Values:")
print(df["owner_count"].isnull().sum())

print("\nSummary:")
print(df["owner_count"].describe())

Frequency of Owner Count:
owner_count
1    532
2    302
3    126
4     40
Name: count, dtype: int64

Invalid Values (owner_count <= 0):
Empty DataFrame
Columns: [vehicle_id, listing_date, manufacture_year, brand, vehicle_type, battery_capacity_kwh, battery_health_pct, range_km, km_driven, charging_time_hr, fast_charging, owner_count, city, service_history, resale_price, charging_efficiency]
Index: []

Missing Values:
0

Summary:
count    1000.000000
mean        1.674000
std         0.844059
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         4.000000
Name: owner_count, dtype: float64
